In [ ]:
import streamlit as st
import base64
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# ==========================================
# 1. Define the LangGraph State
# ==========================================
class PaperState(TypedDict):
    api_key: str
    marks: int
    subject: str
    grade_class: str
    q_type: str
    comments: str
    image_data: list  # Store base64 encoded images
    generated_paper: str

# ==========================================
# 2. Define Graph Nodes
# ==========================================
def generate_question_paper(state: PaperState):
    """Node that calls Gemini to generate the paper."""
    
    # Initialize the LLM with the user's key
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash", 
        api_key=state["api_key"]
    )
    
    # Construct the prompt text
    prompt_text = f"""
    You are an expert teacher. Generate a question paper strictly based on the provided images (if any) and the following criteria:
    - Subject: {state['subject']}
    - Class/Grade: {state['grade_class']}
    - Total Marks: {state['marks']}
    - Question Type: {state['q_type']}
    - Additional Instructions: {state['comments']}
    
    Format the output clearly with sections, question numbers, and marks per question.
    """
    
    # Build the message content handling both text and multiple images
    message_content = [{"type": "text", "text": prompt_text}]
    
    for img_base64 in state["image_data"]:
        message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"}
        })
        
    message = HumanMessage(content=message_content)
    
    # Generate the response
    response = llm.invoke([message])
    
    return {"generated_paper": response.content}

# ==========================================
# 3. Build the Workflow Graph
# ==========================================
workflow = StateGraph(PaperState)
workflow.add_node("generator", generate_question_paper)
workflow.set_entry_point("generator")
workflow.add_edge("generator", END)
app_graph = workflow.compile()

# ==========================================
# 4. Streamlit UI
# ==========================================
st.set_page_config(page_title="AI Question Paper Generator", layout="wide")
st.title("📄 AI Question Paper Generator")

# Sidebar for API Key
with st.sidebar:
    st.header("Authentication")
    user_api_key = st.text_input("Enter your Google Gemini API Key:", type="password")
    st.caption("Get your key from [Google AI Studio](https://aistudio.google.com/).")

# Main Form
with st.form("paper_form"):
    col1, col2, col3 = st.columns(3)
    
    with col1:
        subject = st.text_input("Subject (e.g., Physics, History)")
    with col2:
        grade_class = st.text_input("Class/Grade (e.g., 10th, University)")
    with col3:
        marks = st.number_input("Total Marks", min_value=1, max_value=200, value=50)
        
    q_type = st.selectbox(
        "Question Type Format", 
        ["All MCQ", "All One Word", "Mixed / Balanced"]
    )
    
    uploaded_files = st.file_uploader(
        "Upload Source Material (Images)", 
        type=["png", "jpg", "jpeg"], 
        accept_multiple_files=True
    )
    
    comments = st.text_area("Optional Comments/Specific Instructions", placeholder="e.g., Make the questions high difficulty, focus on application-based concepts...")
    
    submit_button = st.form_submit_button("Generate Question Paper")

# ==========================================
# 5. Execution Logic
# ==========================================
if submit_button:
    if not user_api_key:
        st.error("Please enter your Gemini API Key in the sidebar.")
    elif not subject or not grade_class:
        st.error("Please fill in the Subject and Class fields.")
    else:
        with st.spinner("Analyzing context and generating paper..."):
            
            # Convert uploaded images to base64 for the API
            b64_images = []
            if uploaded_files:
                for file in uploaded_files:
                    b64_images.append(base64.b64encode(file.read()).decode("utf-8"))
            
            # Prepare inputs for LangGraph
            initial_state = {
                "api_key": user_api_key,
                "marks": marks,
                "subject": subject,
                "grade_class": grade_class,
                "q_type": q_type,
                "comments": comments,
                "image_data": b64_images
            }
            
            # Run the graph
            try:
                result = app_graph.invoke(initial_state)
                
                st.success("Paper Generated Successfully!")
                st.markdown("---")
                st.markdown(result["generated_paper"])
                
                # Add a download button
                st.download_button(
                    label="Download Paper as TXT",
                    data=result["generated_paper"],
                    file_name=f"{subject}_{grade_class}_paper.txt",
                    mime="text/plain"
                )
            except Exception as e:
                st.error(f"An error occurred: {e}")

2026-08-11 16:11:40.233 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-11 16:11:40.234 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-11 16:11:40.262 
  command:

    streamlit run c:\Users\aarya\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-11 16:11:40.263 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-11 16:11:40.264 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-11 16:11:40.265 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-11 16:11:40.265 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn